In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv("../data/processed/df_model_ready.csv")
print("Dataset Loaded")
print(f"  Shape: {df.shape}")

STEP 1: Dataset Loaded
  Shape: (148835, 25)


In [3]:
print(df.columns)

Index(['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue',
       'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Total_Cost',
       'Clicks_log', 'Leads_log', 'Conversions_log', 'Revenue_log',
       'Acquisition_Cost_log', 'Profit_Label', 'channel_email',
       'channel_facebook', 'channel_google', 'channel_instagram',
       'channel_whatsapp', 'channel_youtube', 'Campaign_Type_encoded',
       'Target_Audience_encoded', 'Customer_Segment_encoded'],
      dtype='str')


In [4]:
X = df.drop(columns=[
    'Revenue',
    'Revenue_log',
    'ROI',
    'Profit_Label',
    'Clicks_log',
    'Leads_log',
    'Conversions',
    'Acquisition_Cost',
])
y = df['Revenue']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
 
print("\nTrain-Test Split (80/20)")
print(f"  X_train: {X_train.shape}")
print(f"  X_test : {X_test.shape}")



STEP 3: Train-Test Split (80/20)
  X_train: (119068, 17)
  X_test : (29767, 17)


In [6]:
models = {
    "Linear Regression"    : LinearRegression(),
    "Decision Tree"        : DecisionTreeRegressor(random_state=42),
    "Random Forest"        : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}


In [7]:
print("\nSTEP 4: Training Models...")
results = {}
 
for name, model in models.items():
    print(f"\n  Training: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
 
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)
 
    results[name] = {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "model": model}
 
    print(f"    MAE  : {mae:,.2f}")
    print(f"    MSE  : {mse:,.2f}")
    print(f"    RMSE : {rmse:,.2f}")
    print(f"    R²   : {r2:.4f}")



STEP 4: Training Models...

  Training: Linear Regression
    MAE  : 153,043.45
    MSE  : 41,041,283,593.51
    RMSE : 202,586.48
    R²   : 0.7346

  Training: Decision Tree
    MAE  : 172,578.85
    MSE  : 64,939,426,239.59
    RMSE : 254,832.15
    R²   : 0.5801

  Training: Random Forest
    MAE  : 130,138.89
    MSE  : 32,737,299,912.01
    RMSE : 180,934.52
    R²   : 0.7883


In [ ]:
print("\n" + "=" * 60)
print("Model Comparison")
print(f"\n{'Model':<25} {'MAE':>12} {'RMSE':>12} {'R²':>8}")
print("-" * 60)
for name, res in results.items():
    print(f"{name:<25} {res['MAE']:>12,.2f} {res['RMSE']:>12,.2f} {res['R2']:>8.4f}")
 
best_name = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f"\n  ✅ Best Model: {best_name}  (R² = {results[best_name]['R2']:.4f})")



STEP 5: Model Comparison

Model                              MAE         RMSE       R²
------------------------------------------------------------
Linear Regression           153,043.45   202,586.48   0.7346
Decision Tree               172,578.85   254,832.15   0.5801
Random Forest               130,138.89   180,934.52   0.7883

  ✅ Best Model: Random Forest  (R² = 0.7883)


In [ ]:
print("\nSaving Model...")
 
with open("parameters/regression_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
 
# Save feature column names — needed to align input in Streamlit
feature_cols = X.columns.tolist()
with open("parameters/regression_features.pkl", "wb") as f:
    pickle.dump(feature_cols, f)
 
print(f"  ✅ Saved 'regression_model.pkl'")
print(f"  ✅ Saved 'regression_features.pkl'")
print("\n" + "=" * 60)
print("Regression Model Development Complete! ✅")
print("=" * 60)



STEP 6: Saving Model...
  ✅ Saved 'regression_model.pkl'
  ✅ Saved 'regression_features.pkl'

Regression Model Development Complete! ✅
